In [ ]:
# -*- coding: utf-8 -*-
"""
AllRecipes Web Scraper - Extract and Parse Recipe Data
=========================================================
This module extracts recipe information from AllRecipes.com including:
- Recipe metadata (title, servings, cook times)
- Ingredients and cooking instructions
- Nutritional information (from JSON-LD or UI fallback)

The extracted data is formatted as single-row CSV entries for easy database integration.
"""

import requests, json, re, urllib3, csv
from bs4 import BeautifulSoup
from typing import Any, Dict, List, Optional, Tuple

# Suppress SSL warnings for web scraping
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Example recipe URL for testing
URL = "https://www.allrecipes.com/recipe/128750/chinese-broccoli/"

# HTTP headers to identify the scraper (polite web scraping practice)
UA = {"User-Agent": "recipe-extractor/values-only/1.1 (+contact@example.com)"}

# Define the order of nutrients to extract in a consistent manner
NUTRIENT_ORDER: List[str] = [
    "calories","total_fat","saturated_fat","cholesterol","sodium",
    "total_carbohydrate","dietary_fiber","total_sugars","protein",
    "vitamin_c","calcium","iron","potassium",
]

# Map of canonical nutrient names to their various UI label variations
# Used to match nutrition labels in the HTML UI fallback parsing
LABELS: Dict[str, List[str]] = {
    "calories": ["Calories"],
    "total_fat": ["Total Fat","Fat"],
    "saturated_fat": ["Saturated Fat","Sat Fat"],
    "cholesterol": ["Cholesterol"],
    "sodium": ["Sodium"],
    "total_carbohydrate": ["Total Carbohydrate","Total Carbs","Carbohydrates","Carbs"],
    "dietary_fiber": ["Dietary Fiber","Fiber"],
    "total_sugars": ["Total Sugars","Sugars"],
    "protein": ["Protein"],
    "vitamin_c": ["Vitamin C","Vit C"],
    "calcium": ["Calcium"],
    "iron": ["Iron"],
    "potassium": ["Potassium"],
}

# Map JSON-LD schema keys to canonical nutrient names
# Used to normalize nutrition data extracted from structured JSON-LD
JSONLD_TO_CANON = {
    "calories": "calories",
    "fatContent": "total_fat",
    "saturatedFatContent": "saturated_fat",
    "cholesterolContent": "cholesterol",
    "sodiumContent": "sodium",
    "carbohydrateContent": "total_carbohydrate",
    "fiberContent": "dietary_fiber",
    "sugarContent": "total_sugars",
    "proteinContent": "protein",
}

# ============================================================================
# HELPER FUNCTIONS - Time and Format Conversion
# ============================================================================

def _iso8601_to_minutes(iso: Optional[str]) -> Optional[int]:
    """
    Convert ISO 8601 duration format (e.g., "PT1H30M") to total minutes.
    
    Args:
        iso: ISO 8601 duration string (e.g., "P1DT2H30M")
        
    Returns:
        Total minutes as integer, or None if parsing fails
    """
    if not iso or not isinstance(iso, str): return None
    m = re.search(r"P(?:(\d+)D)?(?:T(?:(\d+)H)?(?:(\d+)M)?)?", iso.strip(), re.I)
    if not m: return None
    d = int(m.group(1) or 0); h = int(m.group(2) or 0); mi = int(m.group(3) or 0)
    return d*24*60 + h*60 + mi

def _minutes_to_hhmm(m: Optional[int]) -> Optional[str]:
    """
    Convert minutes to human-readable time format (e.g., "1h 30m").
    
    Args:
        m: Duration in minutes
        
    Returns:
        Formatted string like "1h 30m" or "30m", or None if input is None
    """
    if m is None: return None
    h, mm = divmod(m, 60)
    return f"{h}h {mm}m" if h else f"{mm}m"

def _parse_servings(yield_field: Any) -> Optional[str]:
    """
    Extract and clean the servings field from recipe data.
    
    Args:
        yield_field: Recipe yield data (can be string, list, or other type)
        
    Returns:
        Cleaned serving string, or None if empty
    """
    if not yield_field: return None
    if isinstance(yield_field, list) and yield_field: yield_field = yield_field[0]
    s = re.sub(r"\s+", " ", str(yield_field).strip())
    return s or None

def _to_text_list(x: Any) -> List[str]:
    """
    Convert various data formats to a list of clean text strings.
    Handles lists, dicts with 'text'/'name' fields, and raw strings.
    
    Args:
        x: Data in various formats (list, dict, string, or mixed)
        
    Returns:
        List of non-empty trimmed strings
    """
    out: List[str] = []
    if not x: return out
    if isinstance(x, list):
        for it in x:
            if isinstance(it, dict): out.append(it.get("text") or it.get("name") or str(it))
            else: out.append(str(it))
    elif isinstance(x, dict):
        out.append(x.get("text") or x.get("name") or str(x))
    else:
        out.append(str(x))
    return [i.strip() for i in out if str(i).strip()]

def _flatten_instructions(instr: Any) -> List[str]:
    """
    Flatten cooking instructions from various nested JSON-LD formats
    into a simple list of instruction strings.
    
    Handles both HowToSection structures and flat instruction lists.
    
    Args:
        instr: Instructions in JSON-LD format (list, dict, or nested structures)
        
    Returns:
        List of instruction steps as strings
    """
    steps: List[str] = []
    if not instr: return steps
    def _t(v: Any) -> Optional[str]:
        if isinstance(v, dict): return v.get("text") or v.get("name")
        return str(v)
    if isinstance(instr, list):
        for item in instr:
            if isinstance(item, dict) and item.get("@type") in ("HowToSection", ["HowToSection"]):
                sub = item.get("itemListElement") or item.get("steps"); steps += _to_text_list(sub)
            else:
                tt = _t(item)
                if tt: steps.append(tt)
    elif isinstance(instr, dict):
        if instr.get("@type") in ("HowToSection", ["HowToSection"]):
            sub = instr.get("itemListElement") or instr.get("steps"); steps += _to_text_list(sub)
        else:
            tt = _t(instr)
            if tt: steps.append(tt)
    else:
        steps += _to_text_list(instr)
    return [s for s in (s.strip() for s in steps) if s]

# ============================================================================
# JSON-LD EXTRACTION - Parse Structured Recipe Data
# ============================================================================

def extract_recipe_jsonld(html: str) -> Optional[Dict[str, Any]]:
    """
    Extract Recipe schema from JSON-LD structured data in HTML.
    
    Searches all <script type="application/ld+json"> tags and returns
    the first Recipe object found.
    
    Args:
        html: Full HTML content of the recipe page
        
    Returns:
        Recipe dict from JSON-LD, or None if not found
    """
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup.find_all("script", type="application/ld+json"):
        if not tag.string: continue
        try:
            data = json.loads(tag.string)
        except Exception:
            continue
        for node in _iter_json_objects(data):
            if isinstance(node, dict):
                t = node.get("@type")
                if t == "Recipe" or (isinstance(t, list) and "Recipe" in t):
                    return node
                me = node.get("mainEntity")
                if isinstance(me, dict):
                    mt = me.get("@type")
                    if mt == "Recipe" or (isinstance(mt, list) and "Recipe" in mt):
                        return me
    return None

def _iter_json_objects(obj: Any):
    """
    Recursively yield all dict and list objects in a nested structure.
    Used to search through JSON-LD data.
    
    Args:
        obj: Any JSON-serializable object
        
    Yields:
        All nested dict and list objects
    """
    yield obj
    if isinstance(obj, dict):
        for v in obj.values():
            if isinstance(v, (dict, list)): yield from _iter_json_objects(v)
    elif isinstance(obj, list):
        for e in obj:
            if isinstance(e, (dict, list)): yield from _iter_json_objects(e)

def _normalize_amount_str(s: Optional[str], *, is_calories: bool=False) -> Optional[str]:
    """
    Normalize nutrition amount strings to consistent format (e.g., "10g", "500kcal").
    
    Args:
        s: Raw amount string (e.g., "10 grams", "500 calories")
        is_calories: If True, output uses "kcal"; otherwise extracts value + unit
        
    Returns:
        Normalized string like "10g" or "500kcal", or None if parsing fails
    """
    if not s: return None
    if is_calories:
        m = re.search(r"([0-9]+(?:\.[0-9]+)?)", s)
        return f"{m.group(1)}kcal" if m else None
    m = re.search(r"([0-9]+(?:\.[0-9]+)?)\s*([a-zA-Zµ]+)", s)
    if not m: return None
    val = m.group(1); unit = m.group(2).lower()
    if unit == "cal": unit = "kcal"
    return f"{val}{unit}"

def normalize_nutrition_jsonld(n: Any) -> Dict[str, str]:
    """
    Extract and normalize nutrition data from JSON-LD schema.
    
    Converts JSON-LD nutrition keys to canonical names and normalizes
    amount strings (e.g., "10g", "500kcal").
    
    Args:
        n: JSON-LD nutrition dict with keys like "fatContent", "calories", etc.
        
    Returns:
        Dict mapping canonical nutrient names to normalized amount strings
    """
    out: Dict[str, str] = {}
    if isinstance(n, dict):
        for k, v in n.items():
            canon = JSONLD_TO_CANON.get(k)
            if canon and v is not None:
                if canon == "calories":
                    out[canon] = _normalize_amount_str(str(v), is_calories=True) or str(v)
                else:
                    out[canon] = _normalize_amount_str(str(v)) or str(v)
    return out

# ============================================================================
# UI FALLBACK - Parse Nutrition Facts from HTML Layout
# ============================================================================

def parse_nutrition_ui_amounts(html: str) -> Dict[str, Optional[str]]:
    """
    Extract nutrition amounts from Nutrition Facts display in HTML.
    
    Used as fallback when JSON-LD nutrition data is incomplete.
    Searches for "Nutrition Facts" header and parses nearby text for amounts.
    
    Args:
        html: Full HTML content of the recipe page
        
    Returns:
        Dict mapping canonical nutrient names to amount strings (e.g., "10g")
    """
    soup = BeautifulSoup(html, "html.parser")
    txt = soup.get_text(" ", strip=True)
    m = re.search(r"\bNutrition Facts\b", txt, re.I)
    if not m: return {}
    start = m.start()
    # Trim to a chunk after the header; normalize spaces (incl. NBSP)
    section = txt[start:start+4000]
    section = re.sub(r"[\s\u00A0]+", " ", section).strip()

    results: Dict[str, Optional[str]] = {c: None for c in NUTRIENT_ORDER}

    # For each nutrient, find its label then extract the amount value
    for canon in NUTRIENT_ORDER:
        pos = None
        for label in LABELS[canon]:
            pat = r"\b" + r"[\s\u00A0]+".join(map(re.escape, label.split())) + r"\b"
            mm = re.search(pat, section, re.I)
            if mm:
                pos = mm.end()
                break
        if pos is None:
            continue

        # Search for amount in a short window after the label (reduces false hits)
        window = section[pos:pos+120]

        if canon == "calories":
            m_amt = re.search(r"\b([0-9]{1,5})\b", window)
            if m_amt:
                results[canon] = f"{m_amt.group(1)}kcal"
            continue

        m_amt = re.search(r"\b([0-9]+(?:\.[0-9]+)?)\s*(g|mg|mcg|µg|kcal|cal|kj)\b", window, re.I)
        if m_amt:
            val = m_amt.group(1)
            unit = m_amt.group(2).lower()
            if unit == "cal": unit = "kcal"
            results[canon] = f"{val}{unit}"

    return results

# ============================================================================
# MAIN EXTRACTION - Fetch and Parse Full Recipe
# ============================================================================

def extract_recipe(url: str) -> Dict[str, Any]:
    """
    Fetch a recipe from AllRecipes.com and extract all relevant data.
    
    Process:
    1. Fetch the recipe page via HTTP
    2. Extract recipe data from JSON-LD schema
    3. Parse prep/cook times from ISO 8601 format
    4. Extract ingredients and cooking instructions
    5. Get nutrition from JSON-LD (primary) or UI (fallback)
    
    Args:
        url: Full URL to an AllRecipes.com recipe page
        
    Returns:
        Dict with keys: url, title, prepTime, cookTime, totalTime, servings,
                       ingredients, directions, nutrition_amounts
                       
    Raises:
        RuntimeError: If Recipe JSON-LD schema is not found
        requests.exceptions.RequestException: If HTTP request fails
    """
    resp = requests.get(url, headers=UA, timeout=30, verify=False)
    resp.raise_for_status()

    j = extract_recipe_jsonld(resp.text)
    if not j: raise RuntimeError("Recipe JSON-LD not found.")

    # Extract basic recipe metadata
    title = j.get("name")
    servings = _parse_servings(j.get("recipeYield"))
    prep = _minutes_to_hhmm(_iso8601_to_minutes(j.get("prepTime")))
    cook = _minutes_to_hhmm(_iso8601_to_minutes(j.get("cookTime")))
    total = _minutes_to_hhmm(_iso8601_to_minutes(j.get("totalTime")))

    # Extract ingredients and directions
    ingredients = _to_text_list(j.get("recipeIngredient"))
    directions  = _flatten_instructions(j.get("recipeInstructions"))

    # Extract nutrition amounts: JSON-LD first, UI as fallback
    jsonld_amounts = normalize_nutrition_jsonld(j.get("nutrition", {}))
    ui_amounts     = parse_nutrition_ui_amounts(resp.text)

    # Merge nutrition data (prefer JSON-LD, use UI if not available)
    merged: Dict[str, Optional[str]] = {}
    for canon in NUTRIENT_ORDER:
        merged[canon] = jsonld_amounts.get(canon) or ui_amounts.get(canon) or None

    return {
        "url": url,
        "title": title,
        "prepTime": prep,
        "cookTime": cook,
        "totalTime": total,
        "servings": servings,
        "ingredients": ingredients,
        "directions": directions,
        "nutrition_amounts": merged,
    }

# ============================================================================
# CSV CONVERSION - Format Recipe Data as Single-Row CSV
# ============================================================================

def to_one_row(data: Dict[str, Any], list_sep: str = " | ") -> Dict[str, Any]:
    """
    Convert recipe data dict to single-row CSV format.
    
    Joins multi-value fields (ingredients, directions) into pipe-separated strings.
    Replaces None/empty values with "not provided".
    
    Args:
        data: Recipe data dict from extract_recipe()
        list_sep: Separator for joining list items (default " | ")
        
    Returns:
        Dict with flat structure suitable for CSV writing
    """
    def nv(v):
        """Null/void handler: return 'not provided' for empty values."""
        if v is None or (isinstance(v, str) and not v.strip()):
            return "not provided"
        if isinstance(v, (list, tuple)) and not v:
            return "not provided"
        return v

    row = {
        "title": nv(data.get("title")),
        "url": nv(data.get("url")),
        "servings": nv(data.get("servings")),
        "prepTime": nv(data.get("prepTime")),
        "cookTime": nv(data.get("cookTime")),
        "totalTime": nv(data.get("totalTime")),
        "ingredients": nv(" | ".join(data.get("ingredients") or [])),
        "directions": nv(" | ".join(f"{i+1}. {s}" for i, s in enumerate(data.get("directions") or [], 1))),
    }
    # Add each nutrient as a separate column
    amounts = data.get("nutrition_amounts", {}) or {}
    for canon in NUTRIENT_ORDER:
        row[f"nutrition_{canon}"] = nv(amounts.get(canon))
    return row

def write_one_row_csv(path: str, row: Dict[str, Any], order: Optional[List[str]] = None) -> None:
    """
    Write a single recipe row to a CSV file.
    
    Args:
        path: Output CSV file path
        row: Single recipe row dict (from to_one_row())
        order: Optional list of field names to control column order
    """
    if order is None:
        order = list(row.keys())
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=order)
        w.writeheader(); w.writerow(row)

# ============================================================================
# MAIN ENTRY POINT - Test the extraction pipeline
# ============================================================================

if __name__ == "__main__":
    # Test extraction on a single recipe
    data = extract_recipe(URL)

    print("TITLE:", data["title"])
    print("Servings:", data["servings"])
    print("Times:", data["prepTime"], "/", data["cookTime"], "/", data["totalTime"])
    print("\nNUTRITION (amounts):")
    for k in NUTRIENT_ORDER:
        print(f"- {k}: {data['nutrition_amounts'].get(k) or 'not provided'}")

    # Write to CSV with proper column order
    order = ["title","url","servings","prepTime","cookTime","totalTime","ingredients","directions"] + \
            [f"nutrition_{c}" for c in NUTRIENT_ORDER]
    write_one_row_csv("recipe_single_row.csv", to_one_row(data), order)
    print("\nWrote recipe_single_row.csv with 1 row.")


: 

In [ ]:
# Cell 2: Scrape recipe links from AllRecipes homepage
# Objective: Discover all recipe URLs by crawling the main site

resp = requests.get("https://www.allrecipes.com/", headers=UA, timeout=30, verify=False)
soup = BeautifulSoup(resp.text, "html.parser")
recipe_links = set()

# Find all recipe URLs in the page (links starting with /recipe/)
for a in soup.find_all("a", href=True):
    href = a["href"]
    if href.startswith("https://www.allrecipes.com/recipe/"):
        recipe_links.add(href.split("?")[0])  # Remove query params to avoid duplicates

print(f"Found {len(recipe_links)} recipe URLs.")
for url in list(recipe_links)[:100]:  # Show first 100 as example
    print(url)


In [ ]:
# Cell 3: Empty cell - reserved for future use


In [ ]:
# Cell 4: Test extraction on a single recipe
# Extracts data from the example Chinese Broccoli recipe defined in URL variable

data = extract_recipe("https://www.allrecipes.com/recipe/128750/chinese-broccoli/")


In [ ]:
# Cell 5: Display the extracted recipe data
# Shows the complete structure of the extracted recipe

print(data)


In [ ]:
# Cell 6: Write extracted recipe to CSV file
# Converts the recipe data to single-row CSV format and saves to file

order = ["title","url","servings","prepTime","cookTime","totalTime","ingredients","directions"] + \
            [f"nutrition_{c}" for c in NUTRIENT_ORDER]
write_one_row_csv("recipe_single_row.csv", to_one_row(data), order)


In [ ]:
# Cell 7: Batch extract recipes from discovered links
# Iterates through recipe_links and extracts data from each, handling errors gracefully

all_data = []
for url in recipe_links:
    try:
        data = extract_recipe(url)
        all_data.append(data)
    except Exception as e:
        print(f"Failed to extract {url}: {e}")
print(f"Extracted data from {len(all_data)} recipes.")


In [ ]:
# Cell 8: Write all extracted recipes to CSV file
# Outputs all recipe data to all_recipes_single_row.csv with proper column ordering

order = ["title","url","servings","prepTime","cookTime","totalTime","ingredients","directions"] + \
    [f"nutrition_{c}" for c in NUTRIENT_ORDER]

with open("all_recipes_single_row.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=order)
    writer.writeheader()
    for d in all_data:
        writer.writerow(to_one_row(d))


In [ ]:
# Cell 9: Empty cell - separator/break point


In [ ]:
# Cell 10: Scrape Chinese recipes from a specific cuisine category
# Extracts recipe links from the AllRecipes Chinese cuisine category page

resp = requests.get("https://www.allrecipes.com/recipes/695/world-cuisine/asian/chinese/", headers=UA, timeout=30, verify=False)
soup = BeautifulSoup(resp.text, "html.parser")
links = set()

# Find all recipe URLs on the Chinese recipes page
for a in soup.find_all("a", href=True):
    href = a["href"]
    if href.startswith("https://www.allrecipes.com/recipe/"):
        links.add(href.split("?")[0])  # Remove query params

print(f"Found {len(links)} recipe links.")
for link in list(links)[:100]:  # Show first 100 as example
    print(link)


In [ ]:
# Cell 11: Extract data from all Chinese recipes
# Batch processes the Chinese recipe links, extracting and storing recipe data

all_data = []
for url in links:
    try:
        data = extract_recipe(url)
        all_data.append(data)
    except Exception as e:
        print(f"Failed to extract {url}: {e}")
print(f"Extracted data from {len(all_data)} recipes.")


In [ ]:
# Cell 12: Preview the first extracted Chinese recipe
# Displays the structure of the first recipe in all_data for inspection

print(all_data[0])


In [ ]:
# Cell 13: Write all Chinese recipes to CSV file
# Exports the extracted Chinese recipe data to all_recipes_Chinese.csv

order = ["title","url","servings","prepTime","cookTime","totalTime","ingredients","directions"] + \
    [f"nutrition_{c}" for c in NUTRIENT_ORDER]

with open("all_recipes_Chinese.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=order)
    writer.writeheader()
    for d in all_data:
        writer.writerow(to_one_row(d))


In [ ]:
# Cell 14: Discover all world cuisine category links
# Crawls AllRecipes to find all cuisine category pages (used for comprehensive recipe collection)

resp = requests.get(URL, headers={"User-Agent": 'recipe-extractor/values-only/1.1 (+contact@example.com)'}, timeout=30, verify=False)
soup = BeautifulSoup(resp.text, "html.parser")
cuisine_links = set()

# Find all world-cuisine category links
for a in soup.find_all("a", href=True):
    href = a["href"]
    if href.startswith("https://www.allrecipes.com/recipes/") and "/world-cuisine/" in href:
        cuisine_links.add(href.split("?")[0])

print(f"Found {len(cuisine_links)} cuisine links.")
for link in sorted(cuisine_links):
    print(link)


In [ ]:
# Cell 15: Display all discovered cuisine links
# Shows the complete list of cuisine_links for review

cuisine_links


In [ ]:
# Cell 16: Save cuisine links to CSV
# Exports the discovered cuisine category URLs to cuisine_links.csv for reference

with open("cuisine_links.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["url"])
    for link in cuisine_links:
        writer.writerow([link])


In [ ]:
# Cell 17: Scrape recipe links from all cuisine categories
# Iterates through each cuisine category and collects all unique recipe URLs
# This creates a comprehensive index of all recipes across multiple cuisines

all_recipe_links = set()
for cuisine_url in cuisine_links:
    try:
        resp = requests.get(cuisine_url, headers=UA, timeout=30, verify=False)
        soup = BeautifulSoup(resp.text, "html.parser")
        # Find all recipe URLs on this cuisine page
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if href.startswith("https://www.allrecipes.com/recipe/"):
                all_recipe_links.add(href.split("?")[0])
    except Exception as e:
        print(f"Failed to process {cuisine_url}: {e}")

print(f"Found {len(all_recipe_links)} total recipe links from all cuisines.")
for link in list(all_recipe_links)[:10000]:  # Show first 10000 as example
    print(link)


In [ ]:
# Cell 18: Display and count all recipe links
# Shows the set of all discovered recipe links and their total count

all_recipe_links
print(len(all_recipe_links))


In [ ]:
# Cell 19: Save all recipe links to CSV file
# Persists the comprehensive recipe link index to all_recipe_links.csv for future reference

import csv

with open("all_recipe_links.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["url"])
    for link in all_recipe_links:
        writer.writerow([link])


In [ ]:
# Cell 20: Empty cell - separator before bulk extraction phase


In [ ]:
# Cell 21: Bulk extract recipes from all discovered links
# Major data collection phase: iterates through all_recipe_links and extracts data from each recipe
# Handles errors gracefully to continue processing despite individual failures

all_data = []
for url in all_recipe_links:
    try:
        data = extract_recipe(url)
        all_data.append(data)
    except Exception as e:
        print(f"Failed to extract {url}: {e}")
print(f"Extracted data from {len(all_data)} recipes.")


In [ ]:
# Cell 22: Preview the first extracted recipe
# Displays the complete structure of the first recipe in the bulk extraction for inspection

print(all_data[0])


In [ ]:
# Cell 23: Write all recipes to comprehensive CSV file
# Final output: exports all extracted recipe data to all_recipes.csv
# This is the main dataset combining recipes from all cuisines and categories

order = ["title","url","servings","prepTime","cookTime","totalTime","ingredients","directions"] + \
    [f"nutrition_{c}" for c in NUTRIENT_ORDER]

with open("all_recipes.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=order)
    writer.writeheader()
    for d in all_data:
        writer.writerow(to_one_row(d))


In [ ]:
# Cell 24: Empty cell - reserved for future analysis or post-processing


In [ ]:
# Cell 25: Empty cell - reserved for future analysis or post-processing


In [ ]:
# Cell 26: Empty cell - reserved for future analysis or post-processing
